# Feature-Ablation Test — Is the 99 % Riding on `Destination Port`?
### Dataset: CICIDS-2017  ·  Runs on **Kaggle**

## The question

The leakage sanity-check (negative controls) proved the **training/evaluation harness is
honest** — signal-free data collapses to chance. But it could *not* rule out a subtler issue:
a feature that is **genuinely informative on this dataset but would not generalise** to a real
network.

The prime suspect is **`Destination Port`**. In CICIDS-2017 each attack campaign targeted a
*specific* service: SSH brute-force always hits port 22, the web attacks always hit 80/443,
and so on. A model can therefore get high accuracy by memorising *"port 22 → Brute Force"*
instead of learning the *behaviour* of a brute-force attack. On a real network — where benign
traffic also uses port 22, 80, 443 — that shortcut collapses.

## The test (feature ablation)

Retrain the **identical pipeline** with `Destination Port` removed, and measure how much
performance drops. Interpretation:

| Outcome | Meaning |
|---|---|
| F1 barely drops | The model relies on genuine flow *behaviour*; port was redundant. Strong generalisation evidence. |
| F1 drops moderately | Port helped but behaviour carries most of the signal. Acceptable, report honestly. |
| F1 craters | The model was leaning hard on port memorisation — a real-world red flag. |

We run four configurations so the result is unambiguous:

1. **Full** — all 47 features (baseline, reproduces prior numbers)
2. **No Destination Port** — the key ablation
3. **No Init_Win** — secondary check (Init_Win is endpoint-ish; is *it* a crutch?)
4. **No Port + No Init_Win** — both identifier-like features removed

Each is evaluated on the **real-distribution test set**, on both the binary and multi-class
tasks, with a fast Random Forest (strong enough to exploit any shortcut, fast enough to run
four times).

**Kaggle setup:** attach the FeatureSelection output dataset; set `IN_DIR`. No GPU needed.

## 1. Imports & Config

In [ ]:
import os, json, time, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, recall_score

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 110; plt.rcParams['savefig.bbox'] = 'tight'

RANDOM_SEED = 42; np.random.seed(RANDOM_SEED)
CLASS_NAMES = ['BENIGN','Bot/Infiltration','Brute Force','DDoS','DoS','PortScan','Web Attack']
N_CLASSES = len(CLASS_NAMES)

IN_DIR='/kaggle/input/ids-selected'   # EDIT to your FeatureSelection mount path
OUT_DIR='/kaggle/working'; FIGURES_DIR=os.path.join(OUT_DIR,'figures'); os.makedirs(FIGURES_DIR,exist_ok=True)

_report=[]
def _log(t=''): _report.append(str(t)); print(t)
def _savefig(n,fig=None): p=os.path.join(FIGURES_DIR,n); (fig or plt).savefig(p,dpi=130,bbox_inches='tight'); return p
def write_report():
    p=os.path.join(OUT_DIR,'Ablation_DestinationPort_Report.txt')
    open(p,'w',encoding='utf-8').write('\n'.join(_report)); print('Report ->',p)

_log('='*70); _log('FEATURE-ABLATION TEST (Destination Port)  —  CICIDS-2017')
_log(f'Generated : {datetime.now().strftime("%Y-%m-%d %H:%M")}'); _log('='*70)
print('Setup complete.')

## 2. Load Data

In [ ]:
train_path=os.path.join(IN_DIR,'train_selected.parquet')
test_path =os.path.join(IN_DIR,'test_selected.parquet')
feat_path =os.path.join(IN_DIR,'selected_features.json')
for p in [train_path,test_path,feat_path]:
    if not os.path.exists(p): raise FileNotFoundError(f'{p} not found. Set IN_DIR.')
selected_features=json.load(open(feat_path))['selected_features']
train_df=pd.read_parquet(train_path); test_df=pd.read_parquet(test_path)

# subsample train for speed (RF x 4 configs x 2 tasks); test stays FULL for honest metrics
SAMPLE_N=300_000
tr=(train_df.groupby('label_multi',group_keys=False)
      .apply(lambda g:g.sample(min(len(g),max(1,SAMPLE_N*len(g)//len(train_df))),random_state=RANDOM_SEED)))

PORT_COLS    = ['Destination Port']
INITWIN_COLS = ['Init_Win_bytes_forward','Init_Win_bytes_backward','has_init_win_fwd','has_init_win_bwd']

configs = {
    'Full (47 feats)':        [c for c in selected_features],
    'No Dest Port':           [c for c in selected_features if c not in PORT_COLS],
    'No Init_Win':            [c for c in selected_features if c not in INITWIN_COLS],
    'No Port + No Init_Win':  [c for c in selected_features if c not in PORT_COLS+INITWIN_COLS],
}

_log(''); _log('── SECTION 2 : DATA LOADED ────────────────────────────────')
_log(f'  Train subsample : {len(tr):,} rows   Test (full) : {len(test_df):,} rows')
for name,cols in configs.items():
    _log(f'  {name:<24} -> {len(cols)} features')
_log('')
_log(f'  Dropped in "No Dest Port"  : {PORT_COLS}')
_log(f'  Dropped in "No Init_Win"   : {INITWIN_COLS}')

## 3. Run the Ablation
Same Random Forest, same train/test rows, only the feature set changes. Both tasks per config.

In [ ]:
def fit_eval(cols, task):
    Xtr=tr[cols].values.astype(np.float32); Xte=test_df[cols].values.astype(np.float32)
    ycol='label_binary' if task=='binary' else 'label_multi'
    ytr=tr[ycol].values; yte=test_df[ycol].values
    clf=RandomForestClassifier(n_estimators=120,n_jobs=-1,class_weight='balanced',random_state=RANDOM_SEED)
    clf.fit(Xtr,ytr); pred=clf.predict(Xte)
    if task=='binary':
        return {'f1':f1_score(yte,pred),'acc':accuracy_score(yte,pred),'recall':recall_score(yte,pred)}, pred
    return {'macro_f1':f1_score(yte,pred,average='macro'),'acc':accuracy_score(yte,pred),
            'per_class':f1_score(yte,pred,average=None)}, pred

_log(''); _log('── SECTION 3 : RUNNING ABLATION ───────────────────────────')
bin_res, mult_res, mult_pc = {}, {}, {}
for name,cols in configs.items():
    t0=time.time()
    b,_=fit_eval(cols,'binary'); m,_=fit_eval(cols,'multi')
    bin_res[name]=b; mult_res[name]={'macro_f1':m['macro_f1'],'acc':m['acc']}; mult_pc[name]=m['per_class']
    _log(f'  {name:<24} | binary F1={b["f1"]:.4f}  multi macroF1={m["macro_f1"]:.4f}  ({time.time()-t0:.1f}s)')

## 4. Results — How Much Did Performance Drop?

In [ ]:
bin_df=pd.DataFrame(bin_res).T
mult_df=pd.DataFrame(mult_res).T
base_bin=bin_df.loc['Full (47 feats)','f1']
base_mult=mult_df.loc['Full (47 feats)','macro_f1']
bin_df['F1_drop_vs_full']  = base_bin  - bin_df['f1']
mult_df['macroF1_drop_vs_full'] = base_mult - mult_df['macro_f1']

_log(''); _log('── SECTION 4 : RESULTS ────────────────────────────────────')
_log('  BINARY:'); _log(bin_df.to_string())
_log(''); _log('  MULTI-CLASS:'); _log(mult_df.to_string())
print('BINARY'); display(bin_df); print('MULTI-CLASS'); display(mult_df)

# per-class F1 (multi) across configs
pc=pd.DataFrame(mult_pc, index=CLASS_NAMES)
_log(''); _log('  PER-CLASS F1 (multi) across configs:'); _log(pc.to_string())
display(pc)

fig,axes=plt.subplots(1,2,figsize=(20,5)); names=list(configs.keys())
ax=axes[0]
ax.bar(names,[bin_res[n]['f1'] for n in names],color='#1976D2',alpha=0.85,label='binary F1')
ax.bar(names,[mult_res[n]['macro_f1'] for n in names],color='#7B1FA2',alpha=0.6,label='multi macro-F1')
ax.set_ylim(0,1.05); ax.set_title('Performance vs Feature Set',fontweight='bold')
ax.tick_params(axis='x',rotation=20); ax.legend()
for i,n in enumerate(names):
    ax.text(i,bin_res[n]['f1'],f'{bin_res[n]["f1"]:.3f}',ha='center',va='bottom',fontsize=8)
    ax.text(i,mult_res[n]['macro_f1'],f'{mult_res[n]["macro_f1"]:.3f}',ha='center',va='top',fontsize=8,color='white')
ax=axes[1]
pc.T.plot(kind='bar',ax=ax,width=0.8); ax.set_ylim(0,1.05)
ax.set_title('Per-class multi F1 — does any class collapse without port?',fontweight='bold')
ax.tick_params(axis='x',rotation=20); ax.legend(fontsize=7,ncol=2,loc='lower right')
plt.suptitle('Destination-Port Ablation — does removing it break the model?',fontsize=14,fontweight='bold')
plt.tight_layout(); _savefig('01_ablation.png',fig); plt.show()

## 5. Verdict & Save

In [ ]:
drop_port_bin  = bin_df.loc['No Dest Port','F1_drop_vs_full']
drop_port_mult = mult_df.loc['No Dest Port','macroF1_drop_vs_full']
_log(''); _log('='*70); _log('SECTION 5 : VERDICT'); _log('='*70)
_log(f'  Removing Destination Port:')
_log(f'    binary F1 drop      : {drop_port_bin:+.4f}  ({bin_df.loc["Full (47 feats)","f1"]:.4f} -> {bin_df.loc["No Dest Port","f1"]:.4f})')
_log(f'    multi macroF1 drop  : {drop_port_mult:+.4f}  ({base_mult:.4f} -> {mult_df.loc["No Dest Port","macro_f1"]:.4f})')

def band(d):
    if d < 0.01:  return 'NEGLIGIBLE — model relies on genuine flow behaviour, not port memorisation. Strong generalisation evidence.'
    if d < 0.05:  return 'MILD — port helped a little but behaviour carries the signal. Acceptable; report honestly.'
    if d < 0.15:  return 'MODERATE — port is a meaningful crutch. Real-world performance likely lower than test numbers suggest.'
    return 'SEVERE — the model leaned heavily on port memorisation. Test scores overstate real-world performance.'
_log(''); _log('  Multi-class interpretation: '+band(drop_port_mult))
print('\nVerdict (multi-class):', band(drop_port_mult))

bin_df.to_csv(os.path.join(OUT_DIR,'ablation_binary.csv'))
mult_df.to_csv(os.path.join(OUT_DIR,'ablation_multi.csv'))
pc.to_csv(os.path.join(OUT_DIR,'ablation_perclass.csv'))
write_report()
print('Saved ablation CSVs + report.')